In [1]:
import torch
import random
import sys
import os
import re
import numpy as np


In [2]:
#loading modified skf file
Skf_File = os.path.join("../corrupt_pp", "C-C.skf")

# number of dftb training sets to generate and constraints for the random sampling of the training sets
N_SAMPLES=10
LOWER = -0.6
UPPER = 0.0

In [3]:
# automatically modify the skf file with random values for Ep, Es and SPE
# remark : actually, over 20 parameters should be modified, but for the sake of simplicity, we will only modify 3 parameters in this example. 0ver 20 parameters can not be optimized with BO, as I know.
def modify_skf(skf_file, output_file, Ep, Es, SPE):
    with open(skf_file, "r") as file:
            lines = file.readlines()
    line = lines[1]
    parameters = list(re.finditer( r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", line ))
    replacements= [f"{Ep:.10f}", f"{Es:.10f}", f"{SPE:.10f}"]

    for parameter, replacement in reversed(list(zip(parameters[1:4], replacements))): 
          start, end = parameter.span()
          line = line[:start] + replacement + line[end:]
    lines[1] = line
    print("New Random ", Ep, ", ", Es, ", ",SPE)    
    with open(output_file, "w") as f:
          f.writelines(lines)

In [4]:
# generate the training set by modifying the skf file and running dftb+ for each set of parameters
import dataset
from scipy.stats import qmc
sampler = qmc.LatinHypercube(d=3)
l_bounds = [LOWER, LOWER, LOWER]
u_bounds = [UPPER, UPPER, UPPER]

input = []
output = []
for i in range(N_SAMPLES):
    sample = sampler.random(n=1)
    qmc.discrepancy(sample)
    sample = qmc.scale(sample, l_bounds, u_bounds)
    Ep = sample[0, 0]
    Es = sample[0, 1]
    SPE = sample[0, 2]
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE) # new skf file is created with the random values for Ep, Es and SPE
    energy, force = dataset.test_md() # calculate the energy and forces using dftb for the modified skf file
    input_data = np.array([Ep, Es, SPE]) # create the input data array with the random values for Ep, Es and SPE   
    input.append(input_data) # append the input data to the input list for the training set
    output_data = np.array([energy, force]) # create the output data array with the calculated energy and forces
    output.append(output_data) # append the output data to the output list for the training set
input_tensor = torch.tensor(input, dtype=torch.float64) # create the input tensor for the training set
output_tensor = torch.tensor(output, dtype=torch.float64) # create the output tensor for the training set


    

New Random  -0.5361848260016744 ,  -0.034970145137825814 ,  -0.3463717795755409
Force RMSE: 1.045758 eV/Å
Energy RMSE: 69.295730 eV
New Random  -0.2194690029589076 ,  -0.24819819879401622 ,  -0.09671895818022291
Force RMSE: 0.704031 eV/Å
Energy RMSE: 9.168756 eV
New Random  -0.5858870103578961 ,  -0.08715294040438448 ,  -0.3984385557864677
Force RMSE: 1.401378 eV/Å
Energy RMSE: 90.467097 eV
New Random  -0.5507993302739317 ,  -0.22838689852986782 ,  -0.4564768952291204
Force RMSE: 1.382514 eV/Å
Energy RMSE: 78.895926 eV
New Random  -0.4601884932248268 ,  -0.4214549763776339 ,  -0.13489606566128326
Force RMSE: 0.876608 eV/Å
Energy RMSE: 54.394554 eV
New Random  -0.5689264024686135 ,  -0.479054485138367 ,  -0.3307691894627384
Force RMSE: 1.908361 eV/Å
Energy RMSE: 94.678267 eV
New Random  -0.3900321133317042 ,  -0.5099990647065309 ,  -0.07796017603333394
Force RMSE: 0.410456 eV/Å
Energy RMSE: 39.895757 eV
New Random  -0.09850905392082909 ,  -0.58234606763529 ,  -0.4092562276823225
Force R

/tmp/ipykernel_254358/4153083363.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  input_tensor = torch.tensor(input, dtype=torch.float64) # create the input tensor for the training set


In [5]:
print("Input Tensor: ", input_tensor)
print("Output Tensor: ", output_tensor)


Input Tensor:  tensor([[-0.5362, -0.0350, -0.3464],
        [-0.2195, -0.2482, -0.0967],
        [-0.5859, -0.0872, -0.3984],
        [-0.5508, -0.2284, -0.4565],
        [-0.4602, -0.4215, -0.1349],
        [-0.5689, -0.4791, -0.3308],
        [-0.3900, -0.5100, -0.0780],
        [-0.0985, -0.5823, -0.4093],
        [-0.2161, -0.5150, -0.1028],
        [-0.2358, -0.0021, -0.1248]], dtype=torch.float64)
Output Tensor:  tensor([[69.2957,  1.0458],
        [ 9.1688,  0.7040],
        [90.4671,  1.4014],
        [78.8959,  1.3825],
        [54.3946,  0.8766],
        [94.6783,  1.9084],
        [39.8958,  0.4105],
        [ 7.2015,  0.4635],
        [ 4.2711,  0.1194],
        [13.2458,  0.9267]], dtype=torch.float64)


In [6]:
# BO

bounds = torch.tensor([[LOWER,LOWER,LOWER], [UPPER,UPPER,UPPER]], dtype=torch.float64) # arbitrary bounds for the input parameters Ep, Es and SPE
#normalization and denormalization functions for the input parameters
def normalize(x):
    return (x - bounds[0]) / (bounds[1] - bounds[0])
def denormalize(x):
    return x * (bounds[1] - bounds[0]) + bounds[0]


train_x = normalize(input_tensor)


In [7]:
scale = torch.tensor([0.5, 0.5], dtype=torch.float64) # scale for the output parameters 1:1 Energy and Force

def calculate_loss(outputs):
    # Calculate the loss
    loss = torch.sum((outputs*scale), dim=1, keepdim=True)
    return loss

L = calculate_loss(output_tensor)

train_obj = -L

In [8]:
import os
import torch
import warnings
from typing import Optional

from botorch import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.acquisition.objective import GenericMCObjective

from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.transforms.input import Normalize

from botorch.acquisition import qLogExpectedImprovement, qLogNoisyExpectedImprovement

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

# build model
def generate_model(train_x, train_obj, state_dict = None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        input_transform=Normalize(d=train_x.shape[-1]),
        ).to(train_x)

    model = ModelListGP(model_obj)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    fit_gpytorch_mll(mll)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    print("model is generated")
    return mll, model

mll, model = generate_model(train_x, train_obj)


/home/basic-user/src/pyc/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model is generated


In [9]:

def optimization_observation(acq_func, bounds_, num_restarts=10, raw_samples=512):
    # Optimize acquisition function
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
        new_x, _ = optimize_acqf(
            acq_function=acq_func,
            bounds=bounds_,
            q=1,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )
    new_x = new_x.detach()
    new_x_denormalized = denormalize(new_x)
    print("New x: ", new_x_denormalized)
    Ep, Es, SPE = new_x_denormalized[0].tolist()
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE)
    energy, force = dataset.test_md()
    input_data = np.array([Ep, Es, SPE])    
    output_data = np.array([energy, force])
    input_tensor_new = torch.tensor(input_data.reshape(1,-1), dtype=torch.float64)
    output_tensor_new = torch.tensor(output_data.reshape(1,-1), dtype=torch.float64)
    train_x_new = torch.cat((train_x, normalize(input_tensor_new)), dim=0)
    train_obj_new = torch.cat((train_obj, -calculate_loss(output_tensor_new)), dim=0)
    print("New train_x: ", input_tensor_new)
    print("New train_obj: ", calculate_loss(output_tensor_new))
    return train_x_new, train_obj_new
acqf = qLogExpectedImprovement(model=model, best_f=train_obj.max())
new_x, new_obj = optimization_observation(acqf, bounds)
mll_new, model_new = generate_model(new_x, new_obj)

New x:  tensor([[-0.9600, -0.9600, -0.9600]], dtype=torch.float64)
New Random  -0.96 ,  -0.96 ,  -0.96
Force RMSE: 3.545565 eV/Å
Energy RMSE: 335.462644 eV
New train_x:  tensor([[-0.9600, -0.9600, -0.9600]], dtype=torch.float64)
New train_obj:  tensor([[169.5041]], dtype=torch.float64)
model is generated


In [10]:
print(new_x, new_obj)

tensor([[ 0.1064,  0.9417,  0.4227],
        [ 0.6342,  0.5863,  0.8388],
        [ 0.0235,  0.8547,  0.3359],
        [ 0.0820,  0.6194,  0.2392],
        [ 0.2330,  0.2976,  0.7752],
        [ 0.0518,  0.2016,  0.4487],
        [ 0.3499,  0.1500,  0.8701],
        [ 0.8358,  0.0294,  0.3179],
        [ 0.6398,  0.1417,  0.8286],
        [ 0.6070,  0.9964,  0.7919],
        [-0.6000, -0.6000, -0.6000]], dtype=torch.float64) tensor([[ -35.1707],
        [  -4.9364],
        [ -45.9342],
        [ -40.1392],
        [ -27.6356],
        [ -48.2933],
        [ -20.1531],
        [  -3.8325],
        [  -2.1952],
        [  -7.0862],
        [-169.5041]], dtype=torch.float64)


In [11]:
new_acqf = qLogExpectedImprovement(model=model_new, best_f=train_obj.max())
while np.abs(new_obj.max()) > 0.001:
    new_x, new_obj = optimization_observation(new_acqf, bounds)
    mll_new, model_new = generate_model(new_x, new_obj)

print("x = ", denormalize(new_x), "RSME = ", new_obj)

New x:  tensor([[-0.6000, -0.6000, -0.6000]], dtype=torch.float64)
New Random  -0.6 ,  -0.6 ,  -0.6


/tmp/ipykernel_254358/1369630982.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


Force RMSE: 2.251318 eV/Å
Energy RMSE: 113.809113 eV
New train_x:  tensor([[-0.6000, -0.6000, -0.6000]], dtype=torch.float64)
New train_obj:  tensor([[58.0302]], dtype=torch.float64)
model is generated


/tmp/ipykernel_254358/1369630982.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


New x:  tensor([[-0.6000, -0.6000, -0.6000]], dtype=torch.float64)
New Random  -0.6 ,  -0.6 ,  -0.6
Force RMSE: 2.251318 eV/Å
Energy RMSE: 113.809113 eV
New train_x:  tensor([[-0.6000, -0.6000, -0.6000]], dtype=torch.float64)
New train_obj:  tensor([[58.0302]], dtype=torch.float64)
model is generated


/tmp/ipykernel_254358/1369630982.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


... (181 outputs left out)

/tmp/ipykernel_254358/1369630982.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


Force RMSE: 2.251318 eV/Å
Energy RMSE: 113.809113 eV
New train_x:  tensor([[-0.6000, -0.6000, -0.6000]], dtype=torch.float64)
New train_obj:  tensor([[58.0302]], dtype=torch.float64)
model is generated


/tmp/ipykernel_254358/1369630982.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  while np.abs(new_obj.max()) > 0.001:


KeyboardInterrupt: 

In [ ]:
f, e = dataset.test_md('../trans3d-0-1/')
print("Input: ", input_data)
print("Output: ", 0.5*f+0.5*e)

Force RMSE: 0.000000 eV/Å
Energy RMSE: 0.000000 eV
Input:  [-0.01560073 -0.31862159 -0.30448505]
Output:  2.0691260746716837e-16
